***

Preparing Workspace

***

In [ ]:
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import re
from datetime import date
import requests
import ast
import xlwt
from xlwt.Workbook import *
from pandas import ExcelWriter
import xlsxwriter
import time
import functools as ft

# Plotting
import matplotlib.pyplot as plt 
import plotly
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio

In [ ]:
# Define user
user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

## Set file paths
if user == 'jfontes':
    # Git
    path_git = os.path.join(path_users, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')

    # SharePoint
    path_sp   = os.path.join(path_users, 'Sacramento Area Council of Governments\\Regional Monitoring and Reporting - Documents')
    path_agol = os.path.join(path_sp, 'Process Revamp', 'Task 8. Reproduce Progress Report indicators', 'Indicator Data', 'SACOG Data')
    path_main = os.path.join(path_sp, 'Data')

path_config0 = os.path.join(path_git, 'config')
path_config  = os.path.join(path_git, 'Data', 'Housing', 'config')

In [ ]:
path_housing = os.path.join(path_main, 'Vibrant and Inclusive Places', 'Development', 'SACOG Housing Dataset')


In [ ]:
exec(open(os.path.join(path_config0, 'Functions.py')).read())


***

Cost_4

***

In [ ]:
#Concat_HCD_Data

year_start = 2018
year_end   = 2023

years_to_import = range(year_start, year_end+1)

list_df = []

for year in years_to_import:
    df_year = pd.read_excel(os.path.join(path_housing, 'HCD_Summarized_Yearly.xlsx'), sheet_name = str(year))
    df_year['Year'] = year
    list_df.append(df_year)

df_housing = pd.concat(list_df, ignore_index=True)
df_housing = df_housing[~df_housing['JURS_NAME'].str.contains('https')]
df_housing.loc[df_housing['JURS_NAME'].str.contains('COUNTY'), 'JURS_NAME'] = 'UNINCORPORATED'

df_housing

In [ ]:
path_out = os.path.join(path_main, 'Vibrant and Inclusive Places', 'Development', 'Housing Cost')

In [ ]:
gb_jurisdiction = df_housing.groupby(['CNTY_NAME', 'JURS_NAME', 'Year'], as_index=False)
gb_county       = df_housing.groupby(['CNTY_NAME',              'Year'], as_index=False)
gb_mpo          = df_housing.groupby([                          'Year'], as_index=False)

In [ ]:
indicator_name = 'Cost_4'

metrics = ['Total', 'CO_VLI', 'CO_LI', 'CO_MI', 'CO_AMI']

#Jurisdiction level
print('Organizing indicator Cost_4 by Jurisdictions')
df_cost4_a = gb_jurisdiction[metrics].sum() 
display(df_cost4_a.head(5))

#County level 
print('Organizing indicator Cost_4 by Counties')
df_cost4_b = gb_county[metrics].sum()
display(df_cost4_b.head(5))

#MPO level 
print('Organizing indicator Cost_4 by MPO')
df_cost4_c = gb_mpo[metrics].sum()
display(df_cost4_c.head(5))

# # Export
# df_cost4_a.to_excel(os.path.join(path_out, indicator_name + ' RHNA Income', indicator_name + ' Jurisdictions' + '_SACOG HCD Summarized Data.xlsx'), index=False)
# df_cost4_b.to_excel(os.path.join(path_out, indicator_name + ' RHNA Income', indicator_name + ' Counties'      + '_SACOG HCD Summarized Data.xlsx'), index=False)
# df_cost4_c.to_excel(os.path.join(path_out, indicator_name + ' RHNA Income', indicator_name + ' MPO'           + '_SACOG HCD Summarized Data.xlsx'), index=False)

# print(f"Data frames exported to {path_out}")

In [ ]:
df_plot = df_cost4_c.copy()

df_plot = pd.melt(df_plot, id_vars = ['Year']) 
df_plot = df_plot[df_plot['variable'] != 'Total']
df_plot.columns = [col.lower() for col in df_plot.columns]
df_plot['percentage'] = 100*df_plot['value'] / df_plot.groupby(['year'])['value'].transform('sum')
df_plot = df_plot.sort_values(['year', 'variable'], ascending = [False, True])
# df_plot.to_csv(os.path.join(path_agol, indicator_name, indicator_name + '_MPO_HCD_Summarized_Data.csv'), index = False)
display(df_plot.head())


fig = px.line(df_plot, x='year', y='percentage', color='variable', markers=True)
fig.update_layout(title = 'Housing Permit Green Zone Proportion SACOG')

# fig.write_html(os.path.join(path_out, indicator_name, 'plots', indicator_name + '_Green Zone Prop SACOG_line.html'))
fig.show()